<a href="https://colab.research.google.com/github/vladislava-c/process-automation-analyzer/blob/main/Process_to_automation_ideas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install openai gradio -q

In [ ]:
# Each step = (step text, your label, set of acceptable Microsoft tools, or {"-"} for MANUAL)
# Borderline tools are now part of the set, so they count as correct.
TOOLS = ["Power Automate", "Power Apps", "Copilot", "SharePoint", "Outlook", "Teams", "Microsoft Forms", "Power BI"]

LABELS = ["AUTOMATE", "ASSIST", "MANUAL"]

ANSWER_KEY = [
 {"id": 16, "process": "A customer applies for a small business loan online. We check the documents, an analyst writes a recommendation, the credit committee decides, and the customer is told the outcome.",
  "steps": [
   ("Collect loan application and documents online", "AUTOMATE", {"Microsoft Forms", "Power Apps"}),
   ("Check that all required documents are present", "AUTOMATE", {"Power Automate"}),
   ("Draft credit recommendation for the analyst", "ASSIST", {"Copilot"}),
   ("Credit committee decides on the loan", "MANUAL", {"-"}),
   ("Inform customer of the decision", "AUTOMATE", {"Outlook"})]},                      # borderline label: rejections may need ASSIST

 {"id": 17, "process": "Every quarter the system owners must confirm who still needs access. Someone exports the user list, sends it to each owner, the owners confirm who can stay, unusual access is looked at by security, and unconfirmed accounts are removed. Completion rates are reported.",
  "steps": [
   ("Export current user access list", "AUTOMATE", {"Power Automate"}),
   ("Send list to each system owner for review", "AUTOMATE", {"Outlook", "Power Automate", "Teams"}),
   ("Owner confirms who keeps access", "MANUAL", {"-"}),
   ("Flag unusual or high-risk access", "ASSIST", {"Copilot"}),
   ("Remove accounts that were not confirmed", "AUTOMATE", {"Power Automate"}),       # borderline label: MANUAL in many companies
   ("Report review completion rates", "AUTOMATE", {"Power BI"})]},

 {"id": 18, "process": "A customer asks for a copy of their personal data. Compliance verifies who they are, collects the data from the different systems, checks that it contains nobody else's information, approves the release, and makes sure the 30-day deadline is met.",
  "steps": [
   ("Receive and log the request", "AUTOMATE", {"Microsoft Forms", "Outlook"}),
   ("Verify the identity of the requester", "MANUAL", {"-"}),
   ("Collect data from the different systems", "MANUAL", {"-"}),                       # borderline label: AUTOMATE if connectors exist
   ("Check the data for other people's information", "ASSIST", {"Copilot"}),
   ("Approve release of the data package", "MANUAL", {"-"}),
   ("Track the 30-day deadline", "AUTOMATE", {"Power Automate", "SharePoint"})]},

 {"id": 19, "process": "so basically when someone from sales wants a discount over 10% they message me, i look at the customer history, ask my boss if its ok, then tell sales yes or no and put it in the excel",
  "steps": [
   ("Receive discount request from sales", "AUTOMATE", {"Microsoft Forms", "Teams"}),
   ("Review the customer's history", "ASSIST", {"Copilot"}),
   ("Route the request to the manager for approval", "AUTOMATE", {"Power Automate"}),
   ("Manager decides on the discount", "MANUAL", {"-"}),
   ("Tell sales the decision", "AUTOMATE", {"Teams"}),
   ("Record the decision in the spreadsheet", "AUTOMATE", {"SharePoint", "Power Automate"})]},

 {"id": 20, "process": "we get supplier emails all the time with price changes. somebody reads them, updates our price list, and if its a big jump they call the supplier. once a month the buyers meet and talk about which suppliers to push on price",
  "steps": [
   ("Read supplier emails about price changes", "ASSIST", {"Copilot"}),
   ("Update the price list", "AUTOMATE", {"Power Automate", "SharePoint"}),
   ("Flag price increases above a threshold", "AUTOMATE", {"Power Automate"}),
   ("Call the supplier about large increases", "MANUAL", {"-"}),
   ("Summarise price trends for the monthly buyer meeting", "ASSIST", {"Copilot", "Power BI"}),
   ("Decide which suppliers to push on price", "MANUAL", {"-"})]},
]

# Sanity check: every tool in the key must be a real tool name
for p in ANSWER_KEY:
    for step, label, tools in p["steps"]:
        assert label in LABELS, (p["id"], step)
        assert tools <= set(TOOLS) | {"-"}, (p["id"], step, tools)
        assert (label == "MANUAL") == (tools == {"-"}), (p["id"], step)
print("Loaded", len(ANSWER_KEY), "processes,", sum(len(p["steps"]) for p in ANSWER_KEY), "steps")

In [ ]:
from google.colab import userdata
from openai import OpenAI
import json
import pandas as pd

client = OpenAI(api_key=userdata.get("API"))
MODEL = "gpt-4o-mini"      # try a newer small model too and compare the eval scores
TEMPERATURE = 0            # keeps runs comparable while you tune the prompt

# ---------- Stage 1: split a free-text process into steps ----------
SPLIT_INSTRUCTIONS = """
You break a business process description into its individual steps.
- 3 to 8 steps, in the order they happen.
- Each step is one short imperative phrase, e.g. "Check that all required documents are present".
- Keep every human decision (approve, decide, negotiate, sign off) as its own separate step.
- Do not invent steps that the description does not imply.
Also give a short title for the process (max 6 words).
"""

SPLIT_SCHEMA = {"type": "json_schema", "json_schema": {"name": "process_steps", "strict": True, "schema": {
    "type": "object", "additionalProperties": False, "required": ["process", "steps"],
    "properties": {
        "process": {"type": "string"},
        "steps": {"type": "array", "items": {"type": "string"}}}}}}

# ---------- Stage 2: label each step (this is what the eval measures) ----------
LABEL_INSTRUCTIONS = f"""
You analyze business processes for a company that uses the Microsoft landscape.
You get a process description and a numbered list of its steps. For each step:
1. Name the actor: the role that performs the step today, according to the description (for example Customer,
   Sales, Analyst, Manager, Compliance). Use short role names and spell the same role the same way in every step.
   If the description does not say, infer the most likely role. Use "System" only if the description says a
   system already does the step.
2. Write a one-sentence reason.
3. Assign exactly one label:
   - AUTOMATE: rule-based, repeatable, structured data, no judgment needed.
   - ASSIST: AI or software prepares something (a draft, a summary, a classification, a comparison, a proposal) and a human reviews or approves it.
   - MANUAL: a decision, negotiation, investigation, relationship, accountability, or legal sign-off that a person must own.
4. Pick the best-fit tool, using exactly one name from: {", ".join(TOOLS)}. For MANUAL steps the tool is "-".
Return one entry per step, with n equal to the step number.

Labelling rules:
- Use ASSIST, not MANUAL, whenever AI could do the first pass of the work. This includes reading, reviewing,
  screening or summarising text such as emails, records, logs or customer history. The human decision that
  follows is a separate step.
- A check against a fixed rule or threshold is AUTOMATE. A judgment about what looks unusual, risky or relevant is ASSIST.
- Exception: we have no document-extraction tooling. Pulling structured fields (amounts, dates, names) out of
  attached or scanned files such as invoices, receipts, CVs or contracts into a system is MANUAL with tool "-".
  This applies ONLY to that extraction step. Steps that collect, route, store, send, or check the completeness
  of documents are still AUTOMATE.

Tool rules:
- Sending or scheduling email or calendar items: Outlook
- Notifying or messaging people in chat: Teams
- Collecting submissions or survey responses: Microsoft Forms
- Building a custom input screen or app for users: Power Apps
- Routing, approvals, data transfer, triggers, and other system-to-system steps: Power Automate
- Storing, tracking, or publishing documents and lists: SharePoint
- Dashboards and reports: Power BI
- Summarising, drafting, classifying, comparing, or reading text: Copilot

Worked examples (not from the real data):
Process: Employees submit expense claims with receipts, finance checks them against policy, the manager approves, and the claim is paid.
1. Submit expense claim with receipts -> Employee; AUTOMATE, Microsoft Forms
2. Copy amounts and dates from the receipts into the finance system -> Finance; MANUAL, - (field extraction from files)
3. Check the claim against the expense policy -> Finance; ASSIST, Copilot (AI compares, finance confirms)
4. Manager approves the claim -> Manager; MANUAL, -
5. Send the approved claim to payroll -> Finance; AUTOMATE, Power Automate

Process: Customer complaints arrive by email; every week the team reviews them and decides which to escalate.
1. Summarise this week's complaint emails -> Service team; ASSIST, Copilot
2. Decide which complaints to escalate -> Service team; MANUAL, -
3. Remind the team of the weekly review -> Team lead; AUTOMATE, Teams
"""

LABEL_SCHEMA = {"type": "json_schema", "json_schema": {"name": "step_labels", "strict": True, "schema": {
    "type": "object", "additionalProperties": False, "required": ["steps"],
    "properties": {"steps": {"type": "array", "items": {
        "type": "object", "additionalProperties": False,
        "required": ["n", "actor", "reason", "label", "tool"],   # reason before label: the model explains first
        "properties": {
            "n": {"type": "integer"},
            "actor": {"type": "string"},
            "reason": {"type": "string"},
            "label": {"type": "string", "enum": LABELS},
            "tool": {"type": "string", "enum": TOOLS + ["-"]}}}}}}}}


def _call(system, user, schema):
    r = client.chat.completions.create(
        model=MODEL,
        temperature=TEMPERATURE,
        response_format=schema,
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": user}],
    )
    return json.loads(r.choices[0].message.content)


def split_steps(process_text):
    """Returns (title, [step, ...])."""
    data = _call(SPLIT_INSTRUCTIONS, process_text, SPLIT_SCHEMA)
    steps = [s.strip() for s in data["steps"] if s.strip()]
    return data["process"].strip(), steps


def label_steps(process_text, steps):
    """Returns one dict {actor, reason, label, tool} per input step, in the same order (matched on n, not position)."""
    numbered = "\n".join(f"{i+1}. {s}" for i, s in enumerate(steps))
    data = _call(LABEL_INSTRUCTIONS, f"Process: {process_text}\n\nSteps:\n{numbered}", LABEL_SCHEMA)
    by_n = {s["n"]: s for s in data["steps"]}
    out = []
    for i in range(1, len(steps) + 1):
        s = by_n.get(i, {"actor": "", "reason": "model returned no entry for this step", "label": "?", "tool": "?"})
        tool = "-" if s["label"] == "MANUAL" else s["tool"]
        out.append({"actor": s["actor"].strip(), "reason": s["reason"], "label": s["label"], "tool": tool})
    return out

print("Ready:", MODEL)

In [ ]:
# ---------- BPMN 2.0: turn labelled steps into a diagram ----------
# The XML is built in Python from the step objects. The model never writes XML, so the file is always valid.
import html
from xml.sax.saxutils import escape, quoteattr

BPMN_JS = "https://unpkg.com/bpmn-js@17.11.1/dist"    # viewer used to draw the XML in the browser

# To-be view: BPMN task type and colour per label
TASK_TYPE = {"AUTOMATE": "serviceTask",   # gear icon: done by a system
             "ASSIST": "userTask",        # person icon: a person working with software
             "MANUAL": "manualTask"}      # hand icon: done by a person
COLOURS = {"AUTOMATE": ("#E1F5EE", "#0F6E56"),   # (fill, stroke)
           "ASSIST": ("#EEEDFE", "#534AB7"),
           "MANUAL": ("#F1EFE8", "#5F5E5A")}
AUTOMATION_LANE = "Automation"

# Layout in pixels
POOL_X, POOL_Y = 20, 20
LABEL_W = 30            # width of the pool and lane name strips
LANE_H = 140
COL_W = 170
TASK_W, TASK_H = 125, 90
EVENT = 36


def _bounds(x, y, w, h):
    return f'<dc:Bounds x="{x:.0f}" y="{y:.0f}" width="{w:.0f}" height="{h:.0f}" />'


def build_bpmn(title, rows, view="to-be"):
    """rows: list of dicts with step, actor, label, tool (and optionally reason). Returns BPMN 2.0 XML with layout.
    as-is: one lane per actor, plain tasks.
    to-be: AUTOMATE steps move to an Automation lane; task type and colour follow the label.
    The start event always sits with the actor who starts the process."""
    to_be = view == "to-be"

    def actor(r):
        return r["actor"].strip() or "Unassigned"

    def lane_of(r):
        return AUTOMATION_LANE if to_be and r["label"] == "AUTOMATE" else actor(r)

    # Nodes in order: start event, one task per step, end event
    nodes = [{"id": "Start", "kind": "startEvent", "name": "", "lane": actor(rows[0])}]
    for i, r in enumerate(rows, 1):
        name = r["step"]
        if to_be and r["label"] != "MANUAL" and r["tool"] not in ("", "-"):
            name += f" ({r['tool']})"
        nodes.append({"id": f"Task_{i}", "kind": TASK_TYPE.get(r["label"], "task") if to_be else "task",
                      "name": name, "lane": lane_of(r), "label": r["label"], "doc": r.get("reason", "")})
    nodes.append({"id": "End", "kind": "endEvent", "name": "", "lane": lane_of(rows[-1])})

    lanes = list(dict.fromkeys(n["lane"] for n in nodes))           # in order of first appearance
    lane_ids = {name: f"Lane_{i}" for i, name in enumerate(lanes, 1)}
    lane_y = {name: POOL_Y + i * LANE_H for i, name in enumerate(lanes)}
    content_x = POOL_X + 2 * LABEL_W + 30

    # One column per node, vertically centred in its lane
    for col, n in enumerate(nodes):
        w, h = (EVENT, EVENT) if n["kind"].endswith("Event") else (TASK_W, TASK_H)
        cx = content_x + col * COL_W + TASK_W / 2
        cy = lane_y[n["lane"]] + LANE_H / 2
        n.update(x=cx - w / 2, y=cy - h / 2, w=w, h=h, cy=cy)

    # Sequence flows: straight within a lane, an elbow when the lane changes
    flows = []
    for i in range(len(nodes) - 1):
        a, b = nodes[i], nodes[i + 1]
        x1, x2 = a["x"] + a["w"], b["x"]
        if a["cy"] == b["cy"]:
            points = [(x1, a["cy"]), (x2, b["cy"])]
        else:
            mid = (x1 + x2) / 2
            points = [(x1, a["cy"]), (mid, a["cy"]), (mid, b["cy"]), (x2, b["cy"])]
        flows.append({"id": f"Flow_{i + 1}", "src": a["id"], "tgt": b["id"], "points": points})

    # Process (the model)
    proc = ['<bpmn:laneSet id="LaneSet_1">']
    for name in lanes:
        refs = "".join(f"<bpmn:flowNodeRef>{n['id']}</bpmn:flowNodeRef>" for n in nodes if n["lane"] == name)
        proc.append(f'<bpmn:lane id="{lane_ids[name]}" name={quoteattr(name)}>{refs}</bpmn:lane>')
    proc.append("</bpmn:laneSet>")
    for n in nodes:
        inner = f"<bpmn:documentation>{escape(n['doc'])}</bpmn:documentation>" if n.get("doc") else ""
        inner += "".join(f"<bpmn:incoming>{fl['id']}</bpmn:incoming>" for fl in flows if fl["tgt"] == n["id"])
        inner += "".join(f"<bpmn:outgoing>{fl['id']}</bpmn:outgoing>" for fl in flows if fl["src"] == n["id"])
        proc.append(f'<bpmn:{n["kind"]} id="{n["id"]}" name={quoteattr(n["name"])}>{inner}</bpmn:{n["kind"]}>')
    for fl in flows:
        proc.append(f'<bpmn:sequenceFlow id="{fl["id"]}" sourceRef="{fl["src"]}" targetRef="{fl["tgt"]}" />')

    # Diagram (the layout)
    pool_w = content_x + len(nodes) * COL_W - POOL_X
    pool_h = len(lanes) * LANE_H
    di = [f'<bpmndi:BPMNShape id="Pool_di" bpmnElement="Pool" isHorizontal="true">'
          f'{_bounds(POOL_X, POOL_Y, pool_w, pool_h)}</bpmndi:BPMNShape>']
    for name in lanes:
        di.append(f'<bpmndi:BPMNShape id="{lane_ids[name]}_di" bpmnElement="{lane_ids[name]}" isHorizontal="true">'
                  f'{_bounds(POOL_X + LABEL_W, lane_y[name], pool_w - LABEL_W, LANE_H)}</bpmndi:BPMNShape>')
    for n in nodes:
        colour = ""
        if to_be and n.get("label") in COLOURS:
            fill, stroke = COLOURS[n["label"]]
            colour = (f' bioc:fill="{fill}" bioc:stroke="{stroke}"'
                      f' color:background-color="{fill}" color:border-color="{stroke}"')
        di.append(f'<bpmndi:BPMNShape id="{n["id"]}_di" bpmnElement="{n["id"]}"{colour}>'
                  f'{_bounds(n["x"], n["y"], n["w"], n["h"])}</bpmndi:BPMNShape>')
    for fl in flows:
        waypoints = "".join(f'<di:waypoint x="{x:.0f}" y="{y:.0f}" />' for x, y in fl["points"])
        di.append(f'<bpmndi:BPMNEdge id="{fl["id"]}_di" bpmnElement="{fl["id"]}">{waypoints}</bpmndi:BPMNEdge>')

    nl = "\n    "
    pool_name = quoteattr(f"{title or 'Process'} ({view})")
    return f"""<?xml version="1.0" encoding="UTF-8"?>
<bpmn:definitions xmlns:bpmn="http://www.omg.org/spec/BPMN/20100524/MODEL"
  xmlns:bpmndi="http://www.omg.org/spec/BPMN/20100524/DI"
  xmlns:dc="http://www.omg.org/spec/DD/20100524/DC"
  xmlns:di="http://www.omg.org/spec/DD/20100524/DI"
  xmlns:bioc="http://bpmn.io/schema/bpmn/biocolor/1.0"
  xmlns:color="http://www.omg.org/spec/BPMN/non-normative/color/1.0"
  id="Definitions_1" targetNamespace="http://bpmn.io/schema/bpmn">
  <bpmn:collaboration id="Collaboration_1">
    <bpmn:participant id="Pool" name={pool_name} processRef="Process_1" />
  </bpmn:collaboration>
  <bpmn:process id="Process_1" isExecutable="false">
    {nl.join(proc)}
  </bpmn:process>
  <bpmndi:BPMNDiagram id="Diagram_1">
    <bpmndi:BPMNPlane id="Plane_1" bpmnElement="Collaboration_1">
    {nl.join(di)}
    </bpmndi:BPMNPlane>
  </bpmndi:BPMNDiagram>
</bpmn:definitions>
"""


def bpmn_viewer_html(xml, height=None):
    """Returns an <iframe> that draws the BPMN XML with bpmn-js. Works in Gradio and in notebook output."""
    if height is None:
        height = min(650, int(60 + 0.7 * xml.count("<bpmn:lane ") * LANE_H))   # the viewer scales the diagram to fit the width
    xml_js = json.dumps(xml).replace("</", "<\\/")       # safe inside a <script> tag
    page = f"""<!doctype html><html><head><meta charset="utf-8">
<link rel="stylesheet" href="{BPMN_JS}/assets/diagram-js.css">
<link rel="stylesheet" href="{BPMN_JS}/assets/bpmn-js.css">
<style>html, body, #canvas {{ height: 100%; margin: 0; background: #fff; }}</style></head>
<body><div id="canvas"></div>
<script src="{BPMN_JS}/bpmn-navigated-viewer.production.min.js"></script>
<script>
  const viewer = new BpmnJS({{ container: "#canvas" }});
  viewer.importXML({xml_js})
    .then(() => viewer.get("canvas").zoom("fit-viewport"))
    .catch(err => {{ document.body.textContent = "Could not draw the diagram: " + err.message; }});
</script></body></html>"""
    return (f'<iframe srcdoc="{html.escape(page, quote=True)}" '
            f'style="width:100%; height:{height}px; border:1px solid #ddd; border-radius:8px;"></iframe>')

print("BPMN builder ready")

In [ ]:
# Try it on one process from the answer key: label it, then draw as-is and to-be
from IPython.display import HTML, display

sample = next(p for p in ANSWER_KEY if p["id"] == 19)
sample_steps = [s[0] for s in sample["steps"]]
sample_rows = [{"step": step, **m} for step, m in zip(sample_steps, label_steps(sample["process"], sample_steps))]

display(pd.DataFrame(sample_rows)[["step", "actor", "label", "tool"]])
for sample_view in ["as-is", "to-be"]:
    display(HTML(f"<h4>{sample_view}</h4>" + bpmn_viewer_html(build_bpmn("Discount approval", sample_rows, sample_view))))

In [ ]:
N_RUNS = 3   # 29 steps is small: one flipped step moves the score ~3.5 points, so average over runs

rows = []
for run in range(N_RUNS):
    for p in ANSWER_KEY:
        steps = [s[0] for s in p["steps"]]
        try:
            out = label_steps(p["process"], steps)
        except Exception as e:
            print(f"Run {run}, process {p['id']}: {e}")
            continue
        for (step, key_label, key_tools), m in zip(p["steps"], out):
            rows.append({"run": run, "id": p["id"], "step": step,
                         "key_label": key_label, "model_label": m["label"],
                         "key_tools": " / ".join(sorted(key_tools)), "model_tool": m["tool"],
                         "tool_ok": m["tool"] in key_tools,
                         "has_tool": key_tools != {"-"},
                         "reason": m["reason"]})

df = pd.DataFrame(rows)
df["label_ok"] = df["key_label"] == df["model_label"]

tool_rows = df[df["has_tool"]]
tool_rows_label_ok = tool_rows[tool_rows["label_ok"]]   # tool score without double-counting label misses

per_run = df.groupby("run")["label_ok"].mean()
print(f"Label accuracy: {df['label_ok'].mean():.0%}  (per run: {', '.join(f'{x:.0%}' for x in per_run)})")
print(f"Tool accuracy, label correct:  {tool_rows_label_ok['tool_ok'].mean():.0%}  ({len(tool_rows_label_ok)} step-runs)")
print(f"Tool accuracy, all automatable: {tool_rows['tool_ok'].mean():.0%}  ({len(tool_rows)} step-runs)")
print()
print(pd.crosstab(df["key_label"], df["model_label"], rownames=["Your key"], colnames=["Model said"]))

In [ ]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 220)

# Every miss, with how many runs it happened in and the model's reason
misses = df[~df["label_ok"] | (df["has_tool"] & df["label_ok"] & ~df["tool_ok"])]
(misses.groupby(["id", "step", "key_label", "model_label", "key_tools", "model_tool"])
       .agg(runs=("run", "count"), reason=("reason", "first"))
       .reset_index()
       .sort_values(["runs", "id"], ascending=[False, True]))

In [ ]:
import os
import re
import tempfile
import traceback
import gradio as gr

COLUMNS = ["Step", "Actor", "Label", "Tool", "Reason"]
VIEWS = ["To-be (with automation)", "As-is (today)"]


def do_split(text):
    if not text or not text.strip():
        raise gr.Error("Please describe a process first.")
    try:
        title, steps = split_steps(text)
    except Exception as e:
        traceback.print_exc()
        raise gr.Error(f"Could not split the process: {e}") from e
    return title, "\n".join(steps)


def render(title, table, view):
    """Draws the BPMN diagram and builds the downloads from the result table (including the user's edits)."""
    if table is None or len(table) == 0:
        return "", "", None, None
    table = pd.DataFrame(table, columns=COLUMNS).fillna("").astype(str)
    table = table[table["Step"].str.strip() != ""].copy()
    table["Label"] = table["Label"].str.upper().str.strip()
    bad = sorted(set(table["Label"]) - set(LABELS))
    if bad:
        raise gr.Error(f"Unknown label(s) {bad}. Use one of {LABELS}.")

    rows = [{"step": r.Step, "actor": r.Actor, "label": r.Label, "tool": r.Tool, "reason": r.Reason}
            for r in table.itertuples()]
    view_key = "to-be" if view == VIEWS[0] else "as-is"
    xml = build_bpmn(title, rows, view_key)

    folder = tempfile.mkdtemp()
    slug = re.sub(r"[^A-Za-z0-9]+", "_", title or "process").strip("_").lower() or "process"
    bpmn_path = os.path.join(folder, f"{slug}_{view_key}.bpmn")
    with open(bpmn_path, "w", encoding="utf-8") as f:
        f.write(xml)
    csv_path = os.path.join(folder, f"{slug}.csv")
    table.assign(Process=title)[["Process"] + COLUMNS].to_csv(csv_path, index=False)

    counts = table["Label"].value_counts()
    summary = (f"**{title or 'Process'}**: {len(table)} steps · "
               f"{counts.get('AUTOMATE', 0)} automate · {counts.get('ASSIST', 0)} assist · {counts.get('MANUAL', 0)} manual")
    return bpmn_viewer_html(xml), summary, bpmn_path, csv_path


def do_label(text, title, steps_text, view):
    # Uses the same label_steps() that the eval measures
    if not text or not text.strip():
        raise gr.Error("Please describe a process first.")
    steps = [s.strip() for s in (steps_text or "").splitlines() if s.strip()]
    if not steps:                                   # user skipped the split button: split automatically
        title, steps_text = do_split(text)
        steps = steps_text.splitlines()
    try:
        out = label_steps(text, steps)
    except Exception as e:
        traceback.print_exc()
        raise gr.Error(f"Could not label the steps: {e}") from e

    table = pd.DataFrame({"Step": steps,
                          "Actor": [m["actor"] for m in out],
                          "Label": [m["label"] for m in out],
                          "Tool": [m["tool"] for m in out],
                          "Reason": [m["reason"] for m in out]})
    return (title, "\n".join(steps), table) + render(title, table, view)


example = ("A client submits a form to open an account. Staff check that all fields are filled in, "
           "enter the data in the core system, review sanctions screening hits, and send a welcome email.")

with gr.Blocks(title="Process Automation Analyzer") as demo:
    gr.Markdown("# Process Automation Analyzer\nDescribe a process. It is split into steps, each step is labelled "
                "AUTOMATE / ASSIST / MANUAL with a suggested Microsoft tool, and the result is drawn as a BPMN diagram.\n\n"
                "*Text you enter is sent to the OpenAI API. Don't paste confidential details.*")
    box = gr.Textbox(lines=6, label="1. Describe the process", placeholder="Describe your process in plain words...")
    gr.Examples([[example]], inputs=box)
    split_btn = gr.Button("Split into steps")
    with gr.Row():
        title_box = gr.Textbox(label="Title", scale=1)
        steps_box = gr.Textbox(lines=8, label="2. Steps (one per line, edit freely)", scale=3)
    label_btn = gr.Button("Label steps and draw", variant="primary")

    summary_md = gr.Markdown()
    table = gr.Dataframe(headers=COLUMNS, interactive=True, wrap=True,
                         label="3. Result (you can edit Actor, Label or Tool, then redraw)")
    with gr.Row():
        view = gr.Radio(VIEWS, value=VIEWS[0], label="Diagram")
        redraw_btn = gr.Button("Redraw diagram from table")
    diagram = gr.HTML()
    with gr.Row():
        bpmn_file = gr.File(label="BPMN 2.0 file (opens in Camunda Modeler, bpmn.io, Signavio)")
        csv_file = gr.File(label="CSV")

    render_outputs = [diagram, summary_md, bpmn_file, csv_file]
    split_btn.click(do_split, inputs=box, outputs=[title_box, steps_box])
    label_btn.click(do_label, inputs=[box, title_box, steps_box, view],
                    outputs=[title_box, steps_box, table] + render_outputs)
    redraw_btn.click(render, inputs=[title_box, table, view], outputs=render_outputs)
    view.change(render, inputs=[title_box, table, view], outputs=render_outputs)
    box.change(lambda: ("", ""), outputs=[title_box, steps_box])   # new process text -> clear stale steps

demo.launch(debug=True)